# 1: Install libraries

In [ ]:
!pip install -q transformers datasets evaluate accelerate sentencepiece sacrebleu rouge_score bert_score nltk pandas scikit-learn openpyxl indic-nlp-library


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 8.8 MB/s eta 0:00:00


# 2: Import libraries

In [ ]:
import os
import re
import gc
import math
import torch
import random
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback
)

from indicnlp.transliterate.unicode_transliterate import (
    UnicodeIndicTransliterator
)

import evaluate
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet")
nltk.download("omw-1.4")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Device:", device)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


GPU available: True
GPU: Tesla T4
Device: cuda


# 3: Upload dataset

In [ ]:
from google.colab import files

uploaded = files.upload()
csv_path = list(uploaded.keys())[0]

print("Uploaded file:", csv_path)

Saving Counterspeech_dataset_Old_cleaned (1).csv to Counterspeech_dataset_Old_cleaned (1).csv
Uploaded file: Counterspeech_dataset_Old_cleaned (1).csv


# 4: Load dataset

In [ ]:
df = pd.read_csv(csv_path)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (3011, 5)
Columns: ['id', 'offensive_text', 'counterspeech_text', 'counterspeech_type', 'text_type']


,id,offensive_text,counterspeech_text,counterspeech_type,text_type
0,1,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,fact_based,hate_speech
1,2,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...,warning_of_consequences,sexual_harassment
2,3,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।,empathy,hate_speech
3,4,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...,fact_based,gender_abuse
4,5,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।,moral_appeal,abusive_language


# 5: Keep only generation columns

In [ ]:
required_columns = ["offensive_text", "counterspeech_text"]

for col in required_columns:
    if col not in df.columns:
        raise ValueError(f"Missing required column: {col}")

df = df[required_columns].copy()

df.head()

,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 6: Clean text

In [ ]:
def clean_text(text):
    text = str(text)
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    text = text.replace("\u200c", "")
    text = text.replace("\u200d", "")
    return text

df["offensive_text"] = df["offensive_text"].apply(clean_text)
df["counterspeech_text"] = df["counterspeech_text"].apply(clean_text)

df = df.dropna()
df = df.drop_duplicates()
df = df[df["offensive_text"].str.len() > 2]
df = df[df["counterspeech_text"].str.len() > 2]

print("Cleaned dataset shape:", df.shape)
df.head()

Cleaned dataset shape: (3011, 2)


,offensive_text,counterspeech_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 7: Create input and target text

In [ ]:
# Keep Bengali text in the dataframe.
# It will be converted to Devanagari only when preparing IndicBART inputs.
df["input_text"] = df["offensive_text"]
df["target_text"] = df["counterspeech_text"]

df = df[["input_text", "target_text"]]

df.head()


,input_text,target_text
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...
1,"তুই তো বেশ্যা, রাস্তায় খেপ মাইরা বেরাছ",মানুষ কে সম্মান করতে শিখুন। তবেই নিজে সম্মানিত...
2,হিন্দুরাই দেশের সম্পদ পাচার করতেছে,পুরো একটা গোষ্ঠীকে দোষ দেওয়া ঠিক না।
3,মাইয়া মানুষ আবার গাড়ি চালাইতে পারে নাকি!,এই ধারণা ভুল। বর্তমানে অনেক মেয়েরা সফলভাবে গাড়...
4,এই ছাগলের কথা শুনলে মাথাই গরম হয়ে যায়,ভদ্রভাবে কথা বললে আলোচনা আরও সুন্দর হয়।


# 8: Train-validation-test split

In [ ]:
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    shuffle=True
)

valid_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

print("Train size:", len(train_df))
print("Validation size:", len(valid_df))
print("Test size:", len(test_df))

Train size: 2408
Validation size: 301
Test size: 302


# 9: Convert to Hugging Face Dataset

In [ ]:
dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df.reset_index(drop=True)),
    "validation": Dataset.from_pandas(valid_df.reset_index(drop=True)),
    "test": Dataset.from_pandas(test_df.reset_index(drop=True))
})

dataset

DatasetDict({
    train: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_text', 'target_text'],
        num_rows: 302
    })
})

# 10: Load IndicBART

In [ ]:
MODEL_NAME = "ai4bharat/IndicBART"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    do_lower_case=False,
    use_fast=False,
    keep_accents=True
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

BOS_ID = tokenizer._convert_token_to_id_with_added_voc("<s>")
EOS_ID = tokenizer._convert_token_to_id_with_added_voc("</s>")
PAD_ID = tokenizer._convert_token_to_id_with_added_voc("<pad>")
BN_LANG_ID = tokenizer._convert_token_to_id_with_added_voc("<2bn>")

special_ids = {
    "BOS_ID": BOS_ID,
    "EOS_ID": EOS_ID,
    "PAD_ID": PAD_ID,
    "BN_LANG_ID": BN_LANG_ID
}

print("Special token IDs:", special_ids)

if any(token_id is None for token_id in special_ids.values()):
    raise ValueError("One or more IndicBART special tokens were not found.")

if BN_LANG_ID == tokenizer.unk_token_id:
    raise ValueError("<2bn> was mapped to the unknown token.")

model.config.decoder_start_token_id = BN_LANG_ID
model.config.bos_token_id = BOS_ID
model.config.eos_token_id = EOS_ID
model.config.pad_token_id = PAD_ID
model.config.use_cache = False

model.generation_config.decoder_start_token_id = BN_LANG_ID
model.generation_config.bos_token_id = BOS_ID
model.generation_config.eos_token_id = EOS_ID
model.generation_config.pad_token_id = PAD_ID

model.to(device)

print("Model:", MODEL_NAME)
print("Tokenizer vocabulary:", len(tokenizer))
print("Model vocabulary:", model.config.vocab_size)
print("Decoder start token:", model.config.decoder_start_token_id)
print("Device:", device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/832 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.90M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/221 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/398 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/976M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/267 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/976M [00:00<?, ?B/s]

Special token IDs: {'BOS_ID': 64000, 'EOS_ID': 64001, 'PAD_ID': 0, 'BN_LANG_ID': 64003}
Model: ai4bharat/IndicBART
Tokenizer vocabulary: 64014
Model vocabulary: 64014
Decoder start token: 64003
Device: cuda


# 11: Tokenize dataset

In [ ]:
MAX_INPUT_LENGTH = 160
MAX_TARGET_LENGTH = 112

TASK_PREFIX_BN = "ভদ্র ও নিরাপদ প্রতিউত্তর তৈরি করো: "

def bengali_to_devanagari(text):
    return UnicodeIndicTransliterator.transliterate(
        str(text),
        "bn",
        "hi"
    )

def devanagari_to_bengali(text):
    return UnicodeIndicTransliterator.transliterate(
        str(text),
        "hi",
        "bn"
    )

def format_indicbart_input(bengali_text):
    devanagari_text = bengali_to_devanagari(
        TASK_PREFIX_BN + str(bengali_text).strip()
    )
    return f"{devanagari_text} </s> <2bn>"

def format_indicbart_target(bengali_text):
    # The decoder language is selected by decoder_start_token_id=<2bn>.
    # Therefore labels contain target content followed by EOS, but no
    # duplicated <2bn> token.
    devanagari_text = bengali_to_devanagari(
        str(bengali_text).strip()
    )
    return f"{devanagari_text} </s>"

def preprocess_function(examples):
    formatted_inputs = [
        format_indicbart_input(text)
        for text in examples["input_text"]
    ]

    formatted_targets = [
        format_indicbart_target(text)
        for text in examples["target_text"]
    ]

    model_inputs = tokenizer(
        formatted_inputs,
        max_length=MAX_INPUT_LENGTH,
        padding="max_length",
        truncation=True,
        add_special_tokens=False
    )

    target_tokens = tokenizer(
        formatted_targets,
        max_length=MAX_TARGET_LENGTH,
        padding="max_length",
        truncation=True,
        add_special_tokens=False
    )

    labels = [
        [
            token_id if token_id != PAD_ID else -100
            for token_id in sequence
        ]
        for sequence in target_tokens["input_ids"]
    ]

    model_inputs["labels"] = labels
    return model_inputs

tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text"]
)

tokenized_dataset


Map:   0%|          | 0/2408 [00:00<?, ? examples/s]

Map:   0%|          | 0/301 [00:00<?, ? examples/s]

Map:   0%|          | 0/302 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2408
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 301
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 302
    })
})

# 12: Debug tokenization


In [ ]:
sample = tokenized_dataset["train"][0]

input_ids = [
    token_id
    for token_id, attention in zip(
        sample["input_ids"],
        sample["attention_mask"]
    )
    if attention == 1
]

valid_label_ids = [
    token_id for token_id in sample["labels"]
    if token_id != -100
]

decoded_input_dev = tokenizer.decode(
    input_ids,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False
)

decoded_target_dev = tokenizer.decode(
    valid_label_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)

decoded_target_bn = devanagari_to_bengali(decoded_target_dev)

print("IndicBART input in Devanagari:")
print(decoded_input_dev)

print("\nTarget decoded in Devanagari:")
print(decoded_target_dev)

print("\nTarget converted back to Bengali:")
print(decoded_target_bn)

print("\nValid target token count:", len(valid_label_ids))
print("Last valid target token ID:", valid_label_ids[-1])
print("Expected EOS ID:", EOS_ID)

assert len(valid_label_ids) > 1
assert valid_label_ids[-1] == EOS_ID


IndicBART input in Devanagari:
भदर ओ नरपद परतउततर तर कर: नरर एगय यचछ, कनत कथय सट बलल हरर एस आमर चददगषट तल गलगल करब।</s><2bn>

Target decoded in Devanagari:
भनन मतर मनषक हय न कर आलचन करन। असममनजनक भष छड बतरक अनक बश फलपरस हय.</s>

Target converted back to Bengali:
ভনন মতর মনষক হয ন কর আলচন করন। অসমমনজনক ভষ ছড বতরক অনক বশ ফলপরস হয.</s>

Valid target token count: 35
Last valid target token ID: 64001
Expected EOS ID: 64001


# 13: Data collator


In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100
)

# 14: Batch loss test

In [ ]:
from torch.utils.data import DataLoader

for split_name in ["train", "validation", "test"]:
    valid_counts = [
        sum(token_id != -100 for token_id in labels)
        for labels in tokenized_dataset[split_name]["labels"]
    ]

    print(
        split_name,
        "| minimum valid target tokens:", min(valid_counts),
        "| maximum:", max(valid_counts)
    )

    if min(valid_counts) < 2:
        raise ValueError(
            f"{split_name} contains an invalid target."
        )

small_train_dataset = tokenized_dataset["train"].select(
    range(min(4, len(tokenized_dataset["train"])))
)

test_loader = DataLoader(
    small_train_dataset,
    batch_size=1,
    collate_fn=data_collator
)

batch = next(iter(test_loader))
valid_labels = batch["labels"][batch["labels"] != -100]

print("Input shape:", tuple(batch["input_ids"].shape))
print("Label shape:", tuple(batch["labels"].shape))
print("Valid label tokens:", valid_labels.numel())

batch = {key: value.to(device) for key, value in batch.items()}

model.train()
model.zero_grad(set_to_none=True)
outputs = model(**batch)
test_loss = outputs.loss

print("Training-mode batch loss:", float(test_loss.item()))

if not torch.isfinite(test_loss) or float(test_loss.item()) <= 0:
    raise ValueError("Invalid IndicBART batch loss.")

test_loss.backward()

bad_gradients = [
    name
    for name, parameter in model.named_parameters()
    if parameter.grad is not None
    and not torch.isfinite(parameter.grad).all()
]

model.zero_grad(set_to_none=True)
model.eval()

print("Parameters with non-finite gradients:", len(bad_gradients))

if bad_gradients:
    raise FloatingPointError(
        f"Non-finite gradients found: {bad_gradients[:5]}"
    )


train | minimum valid target tokens: 9 | maximum: 60
validation | minimum valid target tokens: 14 | maximum: 57
test | minimum valid target tokens: 13 | maximum: 58
Input shape: (1, 160)
Label shape: (1, 112)
Valid label tokens: 35
Training-mode batch loss: 6.906849384307861
Parameters with non-finite gradients: 0


# 15: Training arguments

In [ ]:
OUTPUT_DIR = "/content/indicbart_bengali_counterspeech_fixed"

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    num_train_epochs=8,

    weight_decay=0.01,
    warmup_ratio=0.10,
    max_grad_norm=1.0,

    predict_with_generate=False,

    fp16=False,

    logging_steps=25,
    logging_nan_inf_filter=False,

    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    report_to="none",
    seed=SEED
)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


# 16: Create trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=3)
    ]
)

# 17: Train IndicBART

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 3, 'bos_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,38.963552,4.477816
2,30.956052,3.574250
3,28.887534,3.243729
4,27.063398,3.080971
5,26.082476,2.979070
6,25.791636,2.920193
7,25.946436,2.886540
8,25.196228,2.878218


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=1208, training_loss=30.182977209028028, metrics={'train_runtime': 1937.7534, 'train_samples_per_second': 9.941, 'train_steps_per_second': 0.623, 'total_flos': 3261606049873920.0, 'train_loss': 30.182977209028028, 'epoch': 8.0})

# 18: Evaluate loss

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch
25.196228,2.878218,8


{'eval_loss': 2.878218173980713}

# 19: Save model

In [ ]:
FINAL_MODEL_DIR = "/content/final_indicbart_bengali_counterspeech_fixed"

trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)

print("Model saved to:", FINAL_MODEL_DIR)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to: /content/final_indicbart_bengali_counterspeech_fixed


# 20: Manual generation test

In [ ]:
model.config.use_cache = True
model.eval()

def clean_generated_bengali(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def generate_counterspeech(bengali_text):
    formatted_input = format_indicbart_input(bengali_text)

    inputs = tokenizer(
        formatted_input,
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        add_special_tokens=False
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        output_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],

            max_new_tokens=72,
            min_new_tokens=5,

            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=1.20,
            length_penalty=1.0,
            early_stopping=True,

            decoder_start_token_id=BN_LANG_ID,
            bos_token_id=BOS_ID,
            eos_token_id=EOS_ID,
            pad_token_id=PAD_ID,

            remove_invalid_values=True,
            renormalize_logits=True
        )

    generated_dev = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    generated_bn = devanagari_to_bengali(generated_dev)
    return clean_generated_bengali(generated_bn)

preview_predictions = []

for i in range(min(20, len(test_df))):
    source = test_df["input_text"].iloc[i]
    reference = test_df["target_text"].iloc[i]
    prediction = generate_counterspeech(source)

    preview_predictions.append(prediction)

    print("Offensive:", source)
    print("Reference:", reference)
    print("Generated:", prediction)
    print("Generated length:", len(prediction.split()))
    print("-" * 80)

unique_rate = len(set(preview_predictions)) / len(preview_predictions)
empty_count = sum(
    not str(text).strip()
    for text in preview_predictions
)

print("Unique preview rate:", unique_rate)
print("Empty preview outputs:", empty_count)

if unique_rate < 0.40:
    print(
        "WARNING: IndicBART may still be producing repetitive outputs."
    )


Offensive: কি মনে করছ, মেয়েরা কিছু বোঝে ?
Reference: এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথে কাজ করছে।
Generated: <2bn> কউক অপমন ন কর নরদষট কথ বল উচত।</s>
Generated length: 9
--------------------------------------------------------------------------------
Offensive: আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক মূল জিনিস এড়িয়ে যাস!
Reference: প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আচরণ উল্লেখ করলে কথা শক্ত হয়।
Generated: <2bn> পরতরক মল জনস এডয যস।</s>
Generated length: 6
--------------------------------------------------------------------------------
Offensive: নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।
Reference: মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ভদ্র ভাষাই ভালো।
Generated: <2bn> এভবক অপমন ন কর নরদষট কথ বল উচত।</s>
Generated length: 9
--------------------------------------------------------------------------------
Offensive: ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি
Reference: একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।
Generated: <2bn> মনষ গলই শকরর মত 

# 21: Generate full test predictions

In [ ]:
test_inputs = test_df["input_text"].tolist()
references = test_df["target_text"].tolist()

predictions = []

for i, source in enumerate(test_inputs):
    prediction = generate_counterspeech(source)
    predictions.append(prediction)

    if i % 50 == 0:
        print(f"Generated {i}/{len(test_inputs)}")

results_df = pd.DataFrame({
    "input_text": test_inputs,
    "reference_counterspeech": references,
    "generated_counterspeech": predictions
})

results_df["generated_length"] = results_df[
    "generated_counterspeech"
].apply(lambda text: len(str(text).split()))

unique_output_rate = (
    results_df["generated_counterspeech"].nunique()
    / len(results_df)
)

empty_output_rate = (
    results_df["generated_counterspeech"]
    .astype(str)
    .str.strip()
    .eq("")
    .mean()
)

print("Unique output rate:", unique_output_rate)
print("Empty output rate:", empty_output_rate)

results_df.head(10)


Generated 0/302
Generated 50/302
Generated 100/302
Generated 150/302
Generated 200/302
Generated 250/302
Generated 300/302
Unique output rate: 0.5463576158940397
Empty output rate: 0.0


,input_text,reference_counterspeech,generated_counterspeech,generated_length
0,"কি মনে করছ, মেয়েরা কিছু বোঝে ?",এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,<2bn> কউক অপমন ন কর নরদষট কথ বল উচত।</s>,9
1,আরে আরে ইবলিশের ছোট ভাই বলিস কি তুই! প্রতারক ম...,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,<2bn> পরতরক মল জনস এডয যস।</s>,6
2,"নষ্ট সে নষ্টদের তো সাথে থাকবে, এটাই স্বভাবিক।","মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",<2bn> এভবক অপমন ন কর নরদষট কথ বল উচত।</s>,9
3,ভারতীয় মানুষ গুলোই শুকরের মত আচরনের জাতি,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,<2bn> মনষ গলই শকরর মত আচরণ কর উচত।</s>,8
4,নাস্তিক ব্লগারদের সাথে তর্ক বিতর্ক করে লাভ নাই...,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,<2bn> করও অপমন ন কর নরদষট কথ বল খবই অপমনজনক।</s>,10
5,মোবাইলগুলো কেড়ে নিয়ে ওই বেডিদের পিছনে লাঠি ভরে...,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,<2bn> কউক দলই শনত হওয এমন কথ বল খবই অপমনজনক।</s>,10
6,ওইটা আমার বাপ আর তোর আম্মুর বোন রেন্ডির ছেলে,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,<2bn> করও অপমন ন কর নরদষট কথ বল খবই অপমনজনক।</s>,10
7,এই মাদকাসক্ত পাটোয়ারীর পিছনে যেসব কথিত তাওহীদ...,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,<2bn> কউক দডদড কর ঠক নয এমন কথ বল উচত।</s>,10
8,এসব জানোয়ারদের মরন হয় না?,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...",<2bn> করও অপমন ন কর নরদষট কথ বল উচত।</s>,9
9,তুই একটা পাঠান সৈনিক গো বীর্যে জম্ম সেটাই প্রম...,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,<2bn> করও পরমণ নয এমন অপমন কর খবই অপমনজনক।</s>,9


# 22: Check generated length

In [ ]:
print(results_df["generated_length"].describe())

results_df[["reference_counterspeech", "generated_counterspeech", "generated_length"]].head(20)

count    302.000000
mean       9.321192
std        1.262603
min        6.000000
25%        9.000000
50%        9.000000
75%       10.000000
max       13.000000
Name: generated_length, dtype: float64


,reference_counterspeech,generated_counterspeech,generated_length
0,এই কথা ঠিক না। বাংলাদেশে অনেক মেয়ে দক্ষতার সাথ...,<2bn> কউক অপমন ন কর নরদষট কথ বল উচত।</s>,9
1,প্রতারক বললে সমস্যার সমাধান হয় না; নির্দিষ্ট আ...,<2bn> পরতরক মল জনস এডয যস।</s>,6
2,"মানুষকে নষ্ট বলে দাগিয়ে দিলে বোঝাপড়া কমে যায়, ...",<2bn> এভবক অপমন ন কর নরদষট কথ বল উচত।</s>,9
3,একটি জাতিকে শুয়োরের সঙ্গে তুলনা করা ঘৃণা ছড়ায়।,<2bn> মনষ গলই শকরর মত আচরণ কর উচত।</s>,8
4,বিশ্বাসভিন্ন মানুষদের নিয়ে এভাবে বিদ্রূপ করা ...,<2bn> করও অপমন ন কর নরদষট কথ বল খবই অপমনজনক।</s>,10
5,এভাবে আঘাতের কথা বলা আরও ক্ষতি বাড়াবে।,<2bn> কউক দলই শনত হওয এমন কথ বল খবই অপমনজনক।</s>,10
6,কারও পরিবার নিয়ে এমন কটাক্ষ খুবই অশালীন এবং কষ...,<2bn> করও অপমন ন কর নরদষট কথ বল খবই অপমনজনক।</s>,10
7,কারও মতকে সমর্থন করলেই তাকে বুদ্ধি প্রতিবন্ধী ...,<2bn> কউক দডদড কর ঠক নয এমন কথ বল উচত।</s>,10
8,"কারও মৃত্যু কামনা করা মানসিক ক্ষতি ডেকে আনে, ত...",<2bn> করও অপমন ন কর নরদষট কথ বল উচত।</s>,9
9,জাতিগত অপমান বাদ দিলে তর্কটা অনেক বেশি পরিষ্কা...,<2bn> করও পরমণ নয এমন অপমন কর খবই অপমনজনক।</s>,9


# 23: Save predictions

In [ ]:
prediction_csv = "/content/indicbart_bengali_fixed_test_predictions.csv"

results_df.to_csv(
    prediction_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved predictions:", prediction_csv)
files.download(prediction_csv)


Saved predictions: /content/indicbart_bengali_fixed_test_predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 24: Metric helper functions

In [ ]:
smooth = SmoothingFunction().method1

def tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def calculate_bleu_scores(predictions, references):
    bleu_1_scores = []
    bleu_2_scores = []
    bleu_3_scores = []
    bleu_4_scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            bleu_1_scores.append(0)
            bleu_2_scores.append(0)
            bleu_3_scores.append(0)
            bleu_4_scores.append(0)
            continue

        bleu_1_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1, 0, 0, 0),
            smoothing_function=smooth
        ))

        bleu_2_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.5, 0.5, 0, 0),
            smoothing_function=smooth
        ))

        bleu_3_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(1/3, 1/3, 1/3, 0),
            smoothing_function=smooth
        ))

        bleu_4_scores.append(sentence_bleu(
            [ref_tokens],
            pred_tokens,
            weights=(0.25, 0.25, 0.25, 0.25),
            smoothing_function=smooth
        ))

    return {
        "BLEU-1": float(np.mean(bleu_1_scores)),
        "BLEU-2": float(np.mean(bleu_2_scores)),
        "BLEU-3": float(np.mean(bleu_3_scores)),
        "BLEU-4": float(np.mean(bleu_4_scores))
    }


def calculate_meteor(predictions, references):
    scores = []

    for pred, ref in zip(predictions, references):
        pred_tokens = tokenize_bn(pred)
        ref_tokens = tokenize_bn(ref)

        if len(pred_tokens) == 0 or len(ref_tokens) == 0:
            scores.append(0)
            continue

        try:
            score = meteor_score([ref_tokens], pred_tokens)
        except:
            score = 0

        scores.append(score)

    return float(np.mean(scores))


def get_ngrams(tokens, n):
    return list(zip(*[tokens[i:] for i in range(n)]))


def calculate_diversity(predictions):
    all_unigrams = []
    all_bigrams = []

    for text in predictions:
        tokens = tokenize_bn(text)
        all_unigrams.extend(tokens)
        all_bigrams.extend(get_ngrams(tokens, 2))

    distinct_1 = len(set(all_unigrams)) / len(all_unigrams) if len(all_unigrams) > 0 else 0
    distinct_2 = len(set(all_bigrams)) / len(all_bigrams) if len(all_bigrams) > 0 else 0

    diversity = (distinct_1 + distinct_2) / 2

    return {
        "Distinct-1": distinct_1,
        "Distinct-2": distinct_2,
        "Diversity": diversity
    }


def normalize_bn_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text


def token_jaccard_similarity(a, b):
    a_tokens = set(tokenize_bn(normalize_bn_text(a)))
    b_tokens = set(tokenize_bn(normalize_bn_text(b)))

    if len(a_tokens) == 0 or len(b_tokens) == 0:
        return 0.0

    intersection = len(a_tokens.intersection(b_tokens))
    union = len(a_tokens.union(b_tokens))

    return intersection / union if union > 0 else 0.0


def calculate_novelty(predictions, train_targets, similarity_threshold=0.70):
    train_targets = [normalize_bn_text(x) for x in train_targets]
    predictions = [normalize_bn_text(x) for x in predictions]

    max_similarities = []
    near_copy_count = 0

    for pred in predictions:
        if len(pred.strip()) == 0:
            max_similarities.append(0.0)
            continue

        similarities = [
            token_jaccard_similarity(pred, train_text)
            for train_text in train_targets
        ]

        max_similarity = max(similarities)
        max_similarities.append(max_similarity)

        if max_similarity >= similarity_threshold:
            near_copy_count += 1

    average_max_similarity = float(np.mean(max_similarities)) if len(max_similarities) > 0 else 0.0
    near_copy_rate = near_copy_count / len(predictions) if len(predictions) > 0 else 0.0

    novelty = 1 - average_max_similarity

    return {
        "Novelty": novelty,
        "Average Max Train Similarity": average_max_similarity,
        "Near Copy Rate": near_copy_rate
    }


bangla_abuse_words = [
    "চুদ",
    "মাগী",
    "হারামি",
    "শালা",
    "কুত্তা",
    "বাল",
    "খানকি",
    "বেশ্যা",
    "মাদারচোদ",
    "বোকাচোদা"
]


def calculate_abuse_word_rate(predictions, abuse_words):
    count = 0

    for text in predictions:
        text = str(text)
        found = any(word in text for word in abuse_words)

        if found:
            count += 1

    abuse_rate = count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Abuse-word rate": abuse_rate,
        "Abusive outputs": count
    }


def calculate_length_stats(predictions):
    lengths = [len(tokenize_bn(text)) for text in predictions]

    empty_count = sum(1 for text in predictions if len(str(text).strip()) == 0)
    empty_rate = empty_count / len(predictions) if len(predictions) > 0 else 0

    return {
        "Average Generated Length": float(np.mean(lengths)),
        "Empty Output Rate": empty_rate
    }

# 25: Bengali ROUGE

In [ ]:
def simple_tokenize_bn(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.split()


def get_ngrams_list(tokens, n):
    if len(tokens) < n:
        return []
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]


def rouge_n_score(pred, ref, n=1):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    pred_ngrams = get_ngrams_list(pred_tokens, n)
    ref_ngrams = get_ngrams_list(ref_tokens, n)

    if len(pred_ngrams) == 0 or len(ref_ngrams) == 0:
        return 0.0

    pred_counts = {}
    ref_counts = {}

    for ng in pred_ngrams:
        pred_counts[ng] = pred_counts.get(ng, 0) + 1

    for ng in ref_ngrams:
        ref_counts[ng] = ref_counts.get(ng, 0) + 1

    overlap = 0

    for ng in ref_counts:
        overlap += min(ref_counts[ng], pred_counts.get(ng, 0))

    recall = overlap / len(ref_ngrams)
    precision = overlap / len(pred_ngrams)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def lcs_length(x, y):
    m, n = len(x), len(y)
    dp = [[0] * (n + 1) for _ in range(m + 1)]

    for i in range(m):
        for j in range(n):
            if x[i] == y[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])

    return dp[m][n]


def rouge_l_score(pred, ref):
    pred_tokens = simple_tokenize_bn(pred)
    ref_tokens = simple_tokenize_bn(ref)

    if len(pred_tokens) == 0 or len(ref_tokens) == 0:
        return 0.0

    lcs = lcs_length(pred_tokens, ref_tokens)

    recall = lcs / len(ref_tokens)
    precision = lcs / len(pred_tokens)

    if precision + recall == 0:
        return 0.0

    return 2 * precision * recall / (precision + recall)


def calculate_custom_rouge(predictions, references):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for pred, ref in zip(predictions, references):
        rouge1_scores.append(rouge_n_score(pred, ref, n=1))
        rouge2_scores.append(rouge_n_score(pred, ref, n=2))
        rougeL_scores.append(rouge_l_score(pred, ref))

    return {
        "rouge1": float(np.mean(rouge1_scores)),
        "rouge2": float(np.mean(rouge2_scores)),
        "rougeL": float(np.mean(rougeL_scores))
    }


rouge_scores = calculate_custom_rouge(predictions, references)

rouge_scores

{'rouge1': 0.0044164262872359574,
 'rouge2': 0.0,
 'rougeL': 0.0044164262872359574}

# 26: BERTScore

In [ ]:
!pip install -q bert_score

from bert_score import score as bert_score

clean_predictions = [
    str(value).strip() if str(value).strip() else "ফাঁকা"
    for value in predictions
]

clean_references = [
    str(value).strip() if str(value).strip() else "ফাঁকা"
    for value in references
]

try:
    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="xlm-roberta-base",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )
except Exception as error:
    print("XLM-R BERTScore failed:", error)
    print("Trying bert-base-multilingual-cased.")

    P, R, F1 = bert_score(
        cands=clean_predictions,
        refs=clean_references,
        model_type="bert-base-multilingual-cased",
        lang="bn",
        rescale_with_baseline=False,
        verbose=True,
        batch_size=16,
        use_fast_tokenizer=True
    )

bertscore_precision = float(P.mean().item())
bertscore_recall = float(R.mean().item())
bertscore_f1 = float(F1.mean().item())

print("BERTScore Precision:", bertscore_precision)
print("BERTScore Recall:", bertscore_recall)
print("BERTScore F1:", bertscore_f1)


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/30 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/19 [00:00<?, ?it/s]

done in 1.03 seconds, 291.99 sentences/sec
BERTScore Precision: 0.8099679350852966
BERTScore Recall: 0.8330046534538269
BERTScore F1: 0.8212963938713074


# 27: Calculate final metrics


In [ ]:
bleu_scores = calculate_bleu_scores(predictions, references)

meteor = calculate_meteor(predictions, references)

diversity_scores = calculate_diversity(predictions)

novelty_scores = calculate_novelty(
    predictions,
    train_df["target_text"].tolist(),
    similarity_threshold=0.70
)

abuse_scores = calculate_abuse_word_rate(
    predictions,
    bangla_abuse_words
)

length_scores = calculate_length_stats(predictions)

final_metrics = {
    "Model": "IndicBART",

    "BLEU-1": bleu_scores["BLEU-1"],
    "BLEU-2": bleu_scores["BLEU-2"],
    "BLEU-3": bleu_scores["BLEU-3"],
    "BLEU-4": bleu_scores["BLEU-4"],

    "ROUGE-1": rouge_scores["rouge1"],
    "ROUGE-2": rouge_scores["rouge2"],
    "ROUGE-L": rouge_scores["rougeL"],

    "BERTScore Precision": bertscore_precision,
    "BERTScore Recall": bertscore_recall,
    "BERTScore F1": bertscore_f1,

    "Diversity": diversity_scores["Diversity"],
    "Distinct-1": diversity_scores["Distinct-1"],
    "Distinct-2": diversity_scores["Distinct-2"],

    "Novelty": novelty_scores["Novelty"],
    "Average Max Train Similarity": novelty_scores["Average Max Train Similarity"],
    "Near Copy Rate": novelty_scores["Near Copy Rate"],

    "Abuse-word rate": abuse_scores["Abuse-word rate"],
    "Abusive outputs": abuse_scores["Abusive outputs"],

    "METEOR": meteor,

    "Average Generated Length": length_scores["Average Generated Length"],
    "Empty Output Rate": length_scores["Empty Output Rate"]
}

metrics_df = pd.DataFrame([final_metrics])

metrics_df


,Model,BLEU-1,BLEU-2,BLEU-3,BLEU-4,ROUGE-1,ROUGE-2,ROUGE-L,BERTScore Precision,BERTScore Recall,...,Distinct-1,Distinct-2,Novelty,Average Max Train Similarity,Near Copy Rate,Abuse-word rate,Abusive outputs,METEOR,Average Generated Length,Empty Output Rate
0,IndicBART,0.003904,0.001304,0.000943,0.000831,0.004416,0.0,0.004416,0.809968,0.833005,...,0.084192,0.173896,0.981392,0.018608,0.0,0.0,0,0.002099,9.321192,0.0


# 28: Save metrics

In [ ]:
metrics_csv = "/content/indicbart_bengali_fixed_final_metrics.csv"

metrics_df.to_csv(
    metrics_csv,
    index=False,
    encoding="utf-8-sig"
)

print("Saved metrics:", metrics_csv)
files.download(metrics_csv)


Saved metrics: /content/indicbart_bengali_fixed_final_metrics.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 29: Zip and download model

In [ ]:
!zip -r /content/final_indicbart_bengali_counterspeech_fixed.zip /content/final_indicbart_bengali_counterspeech_fixed

files.download("/content/final_indicbart_bengali_counterspeech_fixed.zip")


updating: content/final_indicbart_bengali_counterspeech_fixed/ (stored 0%)
updating: content/final_indicbart_bengali_counterspeech_fixed/model.safetensors


zip error: Interrupted (aborting)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>